# M4 lab: storage and coupled carriers, power, hydrogen and heat in one plant

*Energy Systems, Step by Step* · Modeler tier · lab for lesson M4 · about 45 minutes, plus 90 minutes of practice at the end.

You will rebuild the lesson's plant model as one linear program and solve it with the open-source HiGHS solver, through `scipy.optimize.linprog(method="highs")`. `assert` cells check your results against the lesson's reference block.

1. The two weeks: the plant's wind, Prairie Town's price and the town's heat demand, hour by hour.
2. One LP with three balances (power, hydrogen, heat), three stores (battery, tank, heat store) and their limits.
3. Four configurations in two weeks: where coupling earns most, and where little.
4. The electrolyzer's minimum load: two LP stand-ins and the exact mixed-integer program (`scipy.optimize.milp`).
5. Marginal values: the duals of the three balances, and a check that they are unique.

**How to run it.** It runs in seconds.

1. Put this notebook and the four data files in one folder.
2. In Google Colab: *File → Upload notebook*, upload the four files with the folder icon on the left, then *Runtime → Run all*.
3. Locally: Python 3.9 or later with `numpy` and `scipy` 1.9 or later, started in that folder.

| File (download from the lesson page) | What it holds |
|---|---|
| `m04-storage-multienergy.json` | the plant, its devices, the customers, the weeks, the configurations, and the reference results |
| `town.json` | Prairie Town's fleet, for its price of one more MWh (lesson 9) |
| `town_year.json` | the 8,760 hourly shapes of 2025 (Southwest Power Pool, EIA-930), in thousandths of each series' maximum |
| `cf_targets.json` | EIA's 2025 capacity factors (wind 33.8%) |

**The data.** The plant, its customers and every price are fictional and illustrative. Wind and the town's demand follow real 2025 hourly shapes. The data file names each efficiency's source:

| Efficiency | Value | Source |
|---|---|---|
| battery round trip | 0.8259 | DOE's 2022 storage assessment |
| alkaline electrolyzer, minimum load | 15% | IRENA 2020, Table 6 |
| electrolyzer, power per kg | 57 kWh | DOE Record 24005: a proton exchange membrane (PEM) electrolyzer, another type; illustrative here, inside IRENA's 50–78 kWh for alkaline systems |
| heat pump | 3 MWh of heat per MWh | DOE |

**LP relaxation.** Sections 3 and 5 let the electrolyzer run anywhere from 0 to 20 MW: they leave out its 15% minimum load. Section 4 compares that relaxation with the exact on/off model.

**What the asserts check.** Your weekly costs, each configuration's savings, the three minimum-load costs, and the marginal values in three hours.

Schedules are printed, not asserted: other plans of the same cost can run the devices differently.

In [ ]:
import json
from datetime import datetime
from pathlib import Path

import numpy as np
import scipy
from scipy.optimize import Bounds, LinearConstraint, linprog, milp
from scipy.sparse import coo_matrix

major, minor = (int(x) for x in scipy.__version__.split(".")[:2])
assert (major, minor) >= (1, 9), "this lab needs SciPy 1.9 or later (pip install -U scipy)"
FILES = ["m04-storage-multienergy.json", "town.json", "town_year.json", "cf_targets.json"]
for name in FILES:
    assert Path(name).exists(), f"{name} not found: put it next to this notebook (in Colab, upload it with the folder icon)."
D, town, year, targets = (json.load(open(n)) for n in FILES)
REF = D["reference"]
print("NumPy", np.__version__, "· SciPy", scipy.__version__)
print(year["meta"]["source"])

## 1. The two weeks

- **Plant wind**: output $= 100\,s_t^{\,g}$ MW, rounded to 0.1 MW. Here $s_t$ is the 2025 wind shape (thousandths of its maximum), and the power $g$ makes the year average EIA's 33.8% (method "power" of the lesson's profiles).
- **Prairie Town's demand and supply**, as in lesson 9: demand $450$ MW × shape, wind $150$ MW × shape, sun $200$ MW × shape, and a flat 100 MW baseload plant.
- **Prairie Town's price**: the gap left after those sets the cost of one more MWh. \$0 if negative (spilled), \$40 if the combined-cycle covers it, \$100 if the peaker does.
- **Heat demand**: the town's demand shape within the week, scaled to the week's heat peak (15 MW in winter, 3 MW in summer).
- **Hours**: index $i$ of `town_year.json` is the hour ending at `start_utc` + $i$ (UTC); a week starts at 00:00 local time.

In [ ]:
T = 168
def unit(a):
    return np.maximum(np.asarray(a, float), 0) / 1000

def gamma_for(shape, cf):
    # g with mean(s^g) = cf (s > 0 only), by bisection, as the lesson's profiles.js does
    s = unit(shape); pos = s > 0
    f = lambda g: np.where(pos, np.minimum(1, s) ** g, 0).mean()
    lo, hi = 1e-6, 1.0
    while f(hi) > cf:
        lo, hi = hi, hi * 2
    for _ in range(200):
        if hi - lo <= 1e-13 * max(1, hi):
            break
        mid = (lo + hi) / 2
        lo, hi = (mid, hi) if f(mid) > cf else (lo, mid)
    return (lo + hi) / 2

def js_round(x):   # JavaScript's Math.round (halves up), so the rounding matches the page
    return np.floor(np.asarray(x) + 0.5)

def start_index(wk):
    t0 = datetime.fromisoformat(year["start_utc"]); d0 = datetime.fromisoformat(wk["date"] + "T00:00:00")
    return int(round((d0 - t0).total_seconds() / 3600)) + 1 - wk["utcOffsetHours"]

g = gamma_for(year["wind"], targets["targets"]["wind"]["cf"])
s_wind = unit(year["wind"])
plant_wind = np.where(s_wind > 0, D["plant"]["windMW"] * np.minimum(1, s_wind) ** g, 0)
f = town["fleet"]

def prepare(wk):
    i0 = start_index(wk); sl = slice(i0, i0 + T)
    d = js_round(np.asarray(year["demand"][sl]) / 1000 * D["town"]["demandPeakMW"])
    w = js_round(np.asarray(year["wind"][sl]) / 1000 * f["wind"]["mw"]); s = js_round(np.asarray(year["solar"][sl]) / 1000 * f["solar"]["mw"])
    gap = d - f["nuclear"]["mw"] - w - s
    price = np.where(gap < 0, 0, np.where(gap < f["cc"]["mw"], f["cc"]["cost"], f["peaker"]["cost"]))
    raw = np.asarray(year["demand"][sl], float)
    return {"wind": js_round(plant_wind[sl] * 10) / 10, "price": price.astype(float), "heat": js_round(wk["heatPeakMW"] * raw / raw.max() * 100) / 100}

W = {wk["id"]: prepare(wk) for wk in D["weeks"]}
for wk in D["weeks"]:
    x, r = W[wk["id"]], REF["series"][wk["id"]]
    print(f"{wk['label']}: wind {x['wind'].sum():,.0f} MWh, heat {x['heat'].sum():,.0f} MWh, price hours $0/$40/$100: "
          f"{(x['price'] == 0).sum()}/{(x['price'] == 40).sum()}/{(x['price'] == 100).sum()}")
    assert np.abs(x["wind"] - r["wind"]).max() <= 0.051 and np.array_equal(x["price"], r["price"]) and np.abs(x["heat"] - r["heat"]).max() <= 0.0051
print("Both weeks match the lesson's series.")

## 2. One LP, three carriers

Decisions each hour, named as in the code below, the lesson's equation card and its algebra. Over one hour, 1 MW is 1 MWh.

| Decision | Meaning | Unit |
|---|---|---|
| $\mathit{exp}$, $\mathit{imp}$ | grid sales and purchases, each at most 50 | MW |
| $\mathit{cur}$ | wind curtailed | MW |
| $\mathit{bc}$, $\mathit{bd}$, $s$ | battery charge, discharge and level | MW, MW, MWh |
| $\mathit{ez}$ | electrolyzer power | MW |
| $L$, $\mathit{tr}$ | tank level and hydrogen trucked in | kg |
| $\mathit{hp}$ | heat pump power | MW |
| $\mathit{rec}$ | heat recovered from the electrolyzer | MW of heat |
| $\mathit{hin}$, $\mathit{hout}$, $Q$ | heat-store charge, discharge and level | MW of heat, MWh of heat |
| $\mathit{boil}$ | boiler heat | MW of heat |

Coefficients, with their exact values: $k = 1000/57 \approx 17.54$ kg per MWh (`KPM`), $\eta = 0.8259$ (the battery's round trip, `B["eta"]`), $\mathit{COP} = 3$, $\lambda = 1 - 0.98^{1/24}$ (`LAM`).

| Row | Equation | Its dual |
|---|---|---|
| power | $\mathit{exp} + \mathit{bc} + \mathit{ez} + \mathit{hp} + \mathit{cur} - \mathit{imp} - \mathit{bd} = w_t$ | − (marginal value of power, \$/MWh) |
| hydrogen | $k\,\mathit{ez} + \mathit{tr} + L_{t-1} - L_t = 150$ | marginal cost of a kg, \$/kg |
| heat | $\mathit{COP}\,\mathit{hp} + \mathit{rec} - \mathit{hin} + \mathit{hout} + \mathit{boil} = H_t$ | marginal cost of a MWh of heat |
| network | $\mathit{COP}\,\mathit{hp} + \mathit{rec} - \mathit{hin} + \mathit{hout} \le 12$ | |
| recovered heat | $\mathit{rec} \le 0.2\,\mathit{ez}$ | |
| levels | $s_t = s_{t-1} + \eta\,\mathit{bc} - \mathit{bd}$; $Q_t = (1-\lambda)Q_{t-1} + \mathit{hin} - \mathit{hout}$ (cyclic: $t-1$ of hour 0 is hour 167) | |
| bounds | $400 \le L \le 4{,}000$; $0 \le \mathit{ez} \le 20$, or $3 \le \mathit{ez} \le 20$ when the unit must stay on | |

Goal: $\min \sum_t \big[\, p_t (\mathit{imp}_t - \mathit{exp}_t) + 2.50\, \mathit{tr}_t + 15\, \mathit{boil}_t \,\big]$, the week's cost summed over every hour.

A configuration switches devices off by setting their upper bounds to 0.

In [ ]:
NAMES = ["exp", "imp", "cur", "bc", "bd", "s", "ez", "L", "tr", "hp", "rec", "hin", "hout", "Q", "boil"]
NV = len(NAMES)
col = lambda k, t: NAMES.index(k) * T + t
B, E, K, HH, HY = D["battery"], D["electrolyzer"], D["tank"], D["heat"], D["hydrogen"]
KPM = 1000 / E["kWhPerKg"]; LAM = 1 - (1 - HH["lossPerDay"]) ** (1 / 24)

def cfg_of(cid):
    return next(c for c in D["configs"] if c["id"] == cid)

def build(week, cid, min_load="stop", binary=False):
    x, cf = W[week], cfg_of(cid)
    n = NV * T + (T if binary else 0)
    c = np.zeros(n); lo = np.zeros(n); hi = np.full(n, np.inf)
    def set_(k, lo_=None, hi_=None, cost=None):
        for t in range(T):
            j = col(k, t)
            if lo_ is not None: lo[j] = lo_ if np.isscalar(lo_) else lo_[t]
            if hi_ is not None: hi[j] = hi_ if np.isscalar(hi_) else hi_[t]
            if cost is not None: c[j] = cost if np.isscalar(cost) else cost[t]
    set_("exp", hi_=D["plant"]["exportMW"], cost=-x["price"]); set_("imp", hi_=D["plant"]["importMW"], cost=x["price"]); set_("cur", hi_=x["wind"])
    for k, v in (("bc", B["MW"]), ("bd", B["MW"]), ("s", B["MWh"])):
        set_(k, hi_=v if cf["battery"] else 0)
    set_("ez", lo_=E["minLoad"] * E["MW"] if (cf["ez"] and min_load == "on") else 0, hi_=E["MW"] if cf["ez"] else 0)
    set_("L", lo_=K["minKg"], hi_=K["kg"]); set_("tr", cost=HY["truckPerKg"])
    set_("hp", hi_=HH["pumpMW"] if cf["heat"] else 0); set_("rec", hi_=E["heatShare"] * E["MW"] if cf["rec"] else 0)
    for k in ("hin", "hout"):
        set_(k, hi_=2 * HH["networkMW"] if cf["heat"] else 0)
    set_("Q", hi_=HH["storeMWh"] if cf["heat"] else 0); set_("boil", cost=HH["boilerPerMWh"])
    if binary:
        lo[NV * T:] = 0; hi[NV * T:] = 1
    eq, ub = ([], [], []), ([], [], [])   # (row, col, value)
    beq, bub, eq_names = [], [], []
    def row(store, rhs_list, terms, rhs):
        r = len(rhs_list)
        for (k, t), v in terms.items():
            store[0].append(r); store[1].append(k if isinstance(k, int) else col(k, t)); store[2].append(v)
        rhs_list.append(rhs)
    for t in range(T):
        p = (t - 1) % T
        row(eq, beq, {("exp", t): 1, ("bc", t): 1, ("ez", t): 1, ("hp", t): 1, ("cur", t): 1, ("imp", t): -1, ("bd", t): -1}, x["wind"][t]); eq_names.append(f"p{t}")
        row(eq, beq, {("ez", t): KPM, ("tr", t): 1, ("L", p): 1, ("L", t): -1}, HY["demandKgPerHour"]); eq_names.append(f"h{t}")
        row(eq, beq, {("hp", t): HH["cop"], ("rec", t): 1, ("hin", t): -1, ("hout", t): 1, ("boil", t): 1}, x["heat"][t]); eq_names.append(f"q{t}")
        row(eq, beq, {("s", t): 1, ("bc", t): -B["eta"], ("bd", t): 1, ("s", p): -1}, 0); eq_names.append(f"sb{t}")
        row(eq, beq, {("Q", t): 1, ("hin", t): -1, ("hout", t): 1, ("Q", p): -(1 - LAM)}, 0); eq_names.append(f"sq{t}")
        row(ub, bub, {("hp", t): HH["cop"], ("rec", t): 1, ("hin", t): -1, ("hout", t): 1}, HH["networkMW"])
        row(ub, bub, {("rec", t): 1, ("ez", t): -E["heatShare"]}, 0)
        if binary and cf["ez"]:
            zt = NV * T + t
            row(ub, bub, {("ez", t): 1, (zt, t): -E["MW"]}, 0)
            row(ub, bub, {(zt, t): E["minLoad"] * E["MW"], ("ez", t): -1}, 0)
    A_eq = coo_matrix((eq[2], (eq[0], eq[1])), shape=(len(beq), n)).tocsr()
    A_ub = coo_matrix((ub[2], (ub[0], ub[1])), shape=(len(bub), n)).tocsr()
    return dict(c=c, A_eq=A_eq, b_eq=np.array(beq, float), A_ub=A_ub, b_ub=np.array(bub, float), lo=lo, hi=hi, eq_names=eq_names, n=n,
                integrality=np.r_[np.zeros(NV * T), np.ones(T if binary else 0)])

def solve_lp(m, b_eq=None):
    res = linprog(m["c"], A_ub=m["A_ub"], b_ub=m["b_ub"], A_eq=m["A_eq"], b_eq=m["b_eq"] if b_eq is None else b_eq,
                  bounds=list(zip(m["lo"], m["hi"])), method="highs")
    assert res.status == 0, res.message
    return res

m = build("winter", "all")
print("LP size:", m["n"], "columns x", m["A_eq"].shape[0] + m["A_ub"].shape[0], "rows")
assert (m["n"], m["A_eq"].shape[0] + m["A_ub"].shape[0]) == (REF["size"]["cols"], REF["size"]["rows"])

## 3. Where coupling earns value

Solve every configuration in both weeks (LP relaxation: the electrolyzer may run below its 15% minimum).

- Each configuration adds devices to the one before. A device may sit idle, so the cost cannot rise.
- The saving against "Power only" (the plant with its battery, every kg trucked, all heat from the boiler) is a difference of optimal costs: the same for every best plan.
- These are operating savings. Whether a device pays for itself needs its investment cost, which this model leaves out.

In [ ]:
cost = {}
for wk in D["weeks"]:
    for cf in D["configs"]:
        res = solve_lp(build(wk["id"], cf["id"]))
        cost[(wk["id"], cf["id"])] = res.fun
        assert abs(res.fun - REF["costs"][wk["id"]][cf["id"]]) <= 0.06 + 1e-9 * abs(res.fun), (wk["id"], cf["id"], res.fun)
for cf in [c for c in D["configs"] if not c.get("hidden") and c["id"] != "power"]:
    sav = {w: cost[(w, "power")] - cost[(w, cf["id"])] for w in ("winter", "summer")}
    print(f"{cf['label']:38s} winter ${sav['winter'] / 1e3:6.1f}k   summer ${sav['summer'] / 1e3:6.1f}k")
    assert all(abs(sav[w] - REF["value"][cf["id"]][w]) <= 0.2 for w in sav)
print("battery's value, winter:", round(cost[("winter", "nobat")] - cost[("winter", "power")]))
res = solve_lp(build("winter", "all"))
y = res.x[NAMES.index("ez") * T:(NAMES.index("ez") + 1) * T]
print("Printed, not asserted (one best plan): electrolyzer MWh", round(y.sum()), "· hours below 15%:", int(((y > 1e-4) & (y < 0.15 * E["MW"] - 1e-4)).sum()))

## 4. The minimum load: two LPs and a mixed-integer program

The exact rule needs a binary $z_t$ per hour: $3 z_t \le \mathit{ez}_t \le 20 z_t$.

- Drop it (any load) and the LP is a relaxation; force $z_t = 1$ (always on) and it is a restriction. The exact cost must lie between them.
- `scipy.optimize.milp` solves the exact model with HiGHS. It returns no duals.

In [ ]:
relaxed = cost[("winter", "h2")]
on = solve_lp(build("winter", "h2", min_load="on")).fun
mb = build("winter", "h2", binary=True)
cons = [LinearConstraint(mb["A_eq"], mb["b_eq"], mb["b_eq"]), LinearConstraint(mb["A_ub"], -np.inf, mb["b_ub"])]
ex = milp(mb["c"], constraints=cons, bounds=Bounds(mb["lo"], mb["hi"]), integrality=mb["integrality"])
assert ex.status == 0, ex.message
print(f"relaxed {relaxed:,.0f}  <=  exact {ex.fun:,.0f}  <=  forced on {on:,.0f}  ($ a week)")
assert relaxed <= ex.fun + 0.1 and ex.fun <= on + 0.1
assert abs(ex.fun - REF["binary"]["cost"]) <= 0.1 and abs(on - REF["on"]["cost"]) <= 0.1

## 5. Marginal values, and are they unique?

`linprog` returns `res.eqlin.marginals`: the change of the optimal cost per unit increase of each equality's right-hand side.

- Power row: minus the plant's value of one more MWh of wind. It can differ from the town's price.
- Hydrogen and heat rows: the cost of one more kg, or MWh of heat, demanded.
- At a corner of the cost, several duals fit. So check each against the two one-sided changes of the cost.

In [ ]:
m = build("winter", "all"); base = solve_lp(m); lam = base.eqlin.marginals
def dual(name):
    v = lam[m["eq_names"].index(name)]
    return -v if name.startswith("p") else v
def one_sided(name, d=0.01):
    k = m["eq_names"].index(name); sg = -1 if name.startswith("p") else 1
    b_up, b_dn = m["b_eq"].copy(), m["b_eq"].copy(); b_up[k] += sg * d; b_dn[k] -= sg * d
    return (solve_lp(m, b_up).fun - base.fun) / d, (base.fun - solve_lp(m, b_dn).fun) / d
for h in REF["hours"]:
    t = h["t"]; got = {"power": dual(f"p{t}"), "h2": dual(f"h{t}"), "heat": dual(f"q{t}")}
    z = {k: (0.0 if abs(v) < 0.005 else v) for k, v in got.items()}   # print -0.00 as 0.00
    print(f"{h['label']}: town's price ${W['winter']['price'][t]:.0f}/MWh; power to the plant ${z['power']:.2f}/MWh, hydrogen ${z['h2']:.2f}/kg, heat ${z['heat']:.2f}/MWh")
    for key, row in (("power", "p"), ("h2", "h"), ("heat", "q")):
        up, dn = one_sided(f"{row}{t}")
        assert abs(up - dn) < 1e-4 and abs(up - got[key]) < 1e-3, (h["label"], key, up, dn, got[key])
        assert abs(got[key] - h[key]) < 1e-3
print("The lab matches the browser.")

## 6. Practice (about 90 minutes)

Write your own cells below each task. Predict first, then solve, and state each result as a difference of optimal costs.

Re-run the cells above after each task, to restore the data. The check functions in the next cell test your answers.

1. **A wider connection** (about 15 minutes). Raise both grid limits, `D["plant"]["exportMW"]` and `["importMW"]`, from 50 to 100 MW. Which coupling loses most of its value, and why? Check: `check_wider(saving)`.
2. **PEM instead of alkaline** (about 10 minutes). Set `E["minLoad"] = 0.05` and solve section 4's three models again. How far apart are the two LP bounds now? Check: `check_pem(relaxed, exact, on)`.
3. **Size the electrolyzer** (about 25 minutes). In "+ electrolyzer and tank", make its size $X$ a decision: $\mathit{ez}_t \le X$, no upper limit. What size does each week choose, and why is one week a weak basis? Check: `check_size(mw)`.
4. **A price on CO₂** (about 15 minutes). Charge \$50 per tonne: the boiler emits 0.2 t per MWh of heat, trucked hydrogen 10 kg per kg (both illustrative). Which device's value rises most in each week? Check: `check_co2(saving)`.
5. **A corner** (about 10 minutes). In "Power only, no battery", set the winter week's wind in hour 0 to exactly 50 MW, the connection's limit. Compare the two one-sided changes with the solver's dual. Check: `check_corner(up, down)`.
6. **Write it up** (about 15 minutes). In five sentences, report where coupling earns value in this plant, with the numbers, the boundary (fixed sizes, price-taking, one week each) and what the runs cannot show.

The weekly capital cost of task 3: \$2,000 per kW at 4% real over 30 years, an annuity of $2{,}000{,}000 \times 0.04 / (1 - 1.04^{-30})$ per MW-year, × 168 ÷ 8,760 for one week.

**The check functions.** Each compares your numbers with the answers in the reference block (`REF["practice"]`), solved with the lesson's model. Savings are in \$ a week, against "Power only" with the same changes.

| Function | Its argument |
|---|---|
| `check_wider(saving)`, `check_co2(saving)` | `saving[(week, config)]` for the configurations `h2`, `heat` and `all` |
| `check_pem(relaxed, exact, on)` | the winter week's three costs, \$ a week |
| `check_size(mw)` | `mw[week]`, the size each week chooses, in MW |
| `check_corner(up, down)` | the value of one more MWh in that hour, and the cost of one less, \$/MWh |

In [ ]:
P = REF["practice"]
def _near(yours, ref, tol, what):
    assert abs(yours - ref) <= tol, f"{what}: yours {yours:,.2f}, the reference {ref:,.2f}"
def _savings(saving, block, task):
    for cid, by_week in block.items():
        for week, ref in by_week.items():
            _near(saving[(week, cid)], ref, 0.5, f"task {task}, {cid}, {week}")
def check_wider(saving):
    _savings(saving, P["wider"]["value"], 1); print("Task 1 matches the reference.")
def check_pem(relaxed, exact, on):
    for k, v in (("relaxed", relaxed), ("exact", exact), ("on", on)):
        _near(v, P["pem"][k], 0.5, f"task 2, {k}")
    print("Task 2 matches the reference.")
def check_size(mw):
    for week, ref in P["size"]["MW"].items():
        _near(mw[week], ref, 0.02, f"task 3, {week}")
    print("Task 3 matches the reference.")
def check_co2(saving):
    _savings(saving, P["co2"]["value"], 4); print("Task 4 matches the reference.")
def check_corner(up, down):
    _near(up, P["corner"]["up"], 1e-3, "task 5, one more MWh"); _near(down, P["corner"]["down"], 1e-3, "task 5, one less MWh")
    print("Task 5 matches the reference.")
print("Check functions ready: check_wider, check_pem, check_size, check_co2, check_corner.")